In [ ]:
# Source Code 8 
# Script to plot GeoTIFF images that belong to the same group on a common canvas.

conda install gdal -c conda-forge  # Installs the GDAL library from the conda-forge channel.

import os # Provides functions to interact with the operating system.
import cv2 # Imports OpenCV for image processing.
import numpy as np # Imports NumPy for numerical and array operations.
import matplotlib.pyplot as plt # Imports matplotlib for plotting and visualization.
import rasterio # Library for reading and writing raster data.
from rasterio.plot import plotting_extent # Function to get the spatial extent of a raster for plotting.
import time # Provides time-related functions (e.g., for measuring execution time).

folder_path = r'E:\wi\mhp-inspected-duplicate-cleaned-grouped'
output_folder_path = r'E:\wi\mhp-inspected-duplicate-cleaned-grouped-overlayed'

# Check if the directory already exists; if not, create it
if not os.path.exists(output_folder_path):
    os.makedirs(output_folder_path)

tiff_files = [f for f in os.listdir(folder_path) if f.endswith('.tif')]

file_names_with_groups = {}  # Dictionary to store group names and associated file names

for tiff_file in tiff_files:
    file_name = os.path.splitext(tiff_file)[0]  # Remove the file extension
    parts = file_name.split('_')  # Split the file name by underscores
    group = None

    # Iterate through parts to find the group name
    for part in parts:
        if part.startswith('group'):
            group = part.split('group')[1]
            break

    if group:
        file_names_with_groups.setdefault(group, []).append(tiff_file)

# Print the list of file names and their associated group names
for group, file_names in file_names_with_groups.items():
    print(f"Group {group}: {file_names}")

# Load the last saved group or start from the beginning
last_processed_group = 0
    
# Iterate over each group and overlay images
for group, tiff_files in list(file_names_with_groups.items())[last_processed_group:]:
    # Initialize variables to store overall extent for this group
    overall_min_x = float('inf')
    overall_min_y = float('inf')
    overall_max_x = float('-inf')
    overall_max_y = float('-inf')

    # Initialize plot for this group
    fig, ax = plt.subplots(figsize=(10, 10))

    # Iterate over each GeoTIFF image in this group and plot them
    for tiff_file in tiff_files:
        tiff_path = os.path.join(folder_path, tiff_file)

        # Open the GeoTIFF file and get the spatial information
        with rasterio.open(tiff_path) as src:
            # Get the geographic extent of the image
            extent = plotting_extent(src)

            # Update overall extent for this group
            overall_min_x = min(overall_min_x, extent[0])
            overall_min_y = min(overall_min_y, extent[2])
            overall_max_x = max(overall_max_x, extent[1])
            overall_max_y = max(overall_max_y, extent[3])

            # Read the image and geographic information
            img = src.read(1)  # Assuming a single band image

            # Get the affine transformation to correctly position the image
            transform = src.transform

            # Plot the image with correct geographic positioning
            ax.imshow(img, extent=[transform[2], transform[2] + transform[0] * src.width,
                                   transform[5] + transform[4] * src.height, transform[5]],
                      cmap='gray', alpha=1)

    # Set the overall plot extent for this group
    ax.set_xlim(overall_min_x, overall_max_x)
    ax.set_ylim(overall_min_y, overall_max_y)

    # Hide the axes and labels
    ax.axis('off')

    # Set the DPI for the saved image to achieve the desired resolution
    desired_dpi = 900  # Adjust this value to achieve at least 5000 by 5000 pixels
    output_file_path = os.path.join(output_folder_path, f'overlayed_image_group{group}.png')
    plt.savefig(output_file_path, dpi=desired_dpi, bbox_inches='tight', pad_inches=0, transparent=True)

    print(f'Image saved for Group {group} as:', output_file_path)

    # Delay for 5 seconds
    time.sleep(5)